<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/09_Ensemble_Margin_Classification/01_Cherry_Picker_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: Margin-Bleed Classification & Ensemble Arbitration

## The Business Problem: The "Cherry-Picker" Parasite Basket
Retailers use deep discounts (loss leaders) to drive foot traffic, assuming customers will fill the rest of their baskets with high-margin items. However, "cherry-pickers" exploit this by purchasing *only* the deeply discounted items. These transactions are parasitic, they generate negative margins and actively bleed profitability.

Our goal is to build a classification system to identify these transactions. Because penalizing a legitimate shopper is dangerous, we will not rely on a single model. We will build a **Hard Voting Ensemble** (Support Vector Machines, K-Nearest Neighbors, and Decision Trees) to act as a strict executive arbiter.

## Step 1: Target Definition
Before classification, we must engineer our target variable (`Y`). We will analyze the historical transactions to calculate the `Discount_Ratio` of every basket. By observing the distribution of these discounts, we will draw a strict mathematical boundary to label baskets as `Parasitic (1)` or `Profitable (0)`.

In [1]:
!pip install completejourney_py
import pandas as pd
import numpy as np
from completejourney_py import get_data

print("Fetching transaction data...")
transactions = get_data()['transactions']

# 1. Group the data to the Basket level
# We want the total sales and total discounts for every unique shopping trip
baskets = transactions.groupby('basket_id').agg(
    Total_Sales_Value=('sales_value', 'sum'),
    Retail_Discount=('retail_disc', 'sum'),
    Coupon_Discount=('coupon_disc', 'sum'),
    Coupon_Match=('coupon_match_disc', 'sum'),
    Total_Items=('quantity', 'sum')
).reset_index()

# 2. Calculate the Absolute Total Discount
# In this dataset, discounts are recorded as negative numbers, so we take the absolute value
baskets['Total_Discount'] = (
    baskets['Retail_Discount'].abs() +
    baskets['Coupon_Discount'].abs() +
    baskets['Coupon_Match'].abs()
)

# 3. Calculate the Gross Value of the basket (what it would have cost without any sales/coupons)
baskets['Gross_Value'] = baskets['Total_Sales_Value'] + baskets['Total_Discount']

# 4. Calculate the Discount Ratio
# We use np.where to avoid division by zero for $0 baskets
baskets['Discount_Ratio'] = np.where(
    baskets['Gross_Value'] > 0,
    baskets['Total_Discount'] / baskets['Gross_Value'],
    0
)

# 5. Let's observe the distribution to make a data-driven decision
print("\n📊 Statistical Distribution of the Discount Ratio across all baskets:")
percentiles = [0.25, 0.50, 0.75, 0.85, 0.90, 0.95, 0.99]
display(baskets['Discount_Ratio'].describe(percentiles=percentiles))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 36.0 MB/s eta 0:00:00
Fetching transaction data...

📊 Statistical Distribution of the Discount Ratio across all baskets:


,Discount_Ratio
count,155848.000000
mean,0.133888
std,0.120011
min,0.000000
25%,0.036643
50%,0.113183
75%,0.202684
85%,0.253880
90%,0.294833
95%,0.365931


## Explanation:
Looking directly at the 50th percentile (the median). The average shopper saves about 11% on their grocery bill. Even at the 75th percentile, customers are only saving 20%. This represents normal, healthy promotional behavior.

The anomaly is at the 95th percentile.

These shoppers are securing a 36% to 49% discount across their entire transaction. Because grocery margins rarely exceed 3%, any basket with a total discount ratio above 35% is mathematically guaranteed to be a loss for the store.

We now have our data-driven boundary. We will label any basket with a Discount_Ratio > 0.35 as a Parasitic (1) transaction. Everything else is Profitable (0).

## Step 2: Feature Engineering & Target Definition

Based on the statistical distribution of historical transactions, the median discount ratio is 11%, representing normal shopping behavior. However, the top 5% of baskets exhibit discount ratios exceeding 36%.

Given standard retail margins (2-3%), we establish a strict boundary: **Any basket with a discount ratio > 35% is labeled as Parasitic (1).**

To train our models to identify these baskets without explicitly giving them the discount ratio, we engineer structural basket features:
* **Basket_Size:** Total number of items purchased.
* **Gross_Value:** The total pre-discount value of the basket.
* **Avg_Item_Value:** `Gross_Value / Basket_Size` (Cherry-pickers often target high-value, specific items rather than bulk cheap goods).

In [2]:
# 1. Define the Target Variable (Y)
parasite_threshold = 0.35
baskets['Is_Parasite'] = (baskets['Discount_Ratio'] > parasite_threshold).astype(int)

# 2. Engineer the Features (X)
# We want features that describe the structure of the basket, independent of the discounts
baskets['Avg_Item_Value'] = baskets['Gross_Value'] / baskets['Total_Items']

# Filter out erroneous data (e.g., $0 baskets or negative items)
analytical_df = baskets[(baskets['Gross_Value'] > 0) & (baskets['Total_Items'] > 0)].copy()

# 3. Select the final columns for the machine learning pipeline
features = ['Total_Items', 'Gross_Value', 'Avg_Item_Value']
target = 'Is_Parasite'

ml_df = analytical_df[features + [target]].dropna()

print("✅ Dataset Engineered!")
print(f"Total Baskets Processed: {len(ml_df):,}\n")

# 4. Check the Class Balance
class_balance = ml_df[target].value_counts(normalize=True) * 100
print("⚖️ Class Distribution:")
print(f"Profitable (0): {class_balance[0]:.2f}%")
print(f"Parasitic (1):  {class_balance[1]:.2f}%")

✅ Dataset Engineered!
Total Baskets Processed: 155,381

⚖️ Class Distribution:
Profitable (0): 94.22%
Parasitic (1):  5.78%


## Step 3: Algorithmic Baseline Testing

Rather than pre-selecting a complex ensemble, we adopt an empirical, data-first approach. We will test a suite of distinct algorithmic architectures against our unmanipulated, highly imbalanced dataset (94% Profitable / 6% Parasitic).

Our primary evaluation metrics are focused strictly on the minority class:
* **Precision:** If the model flags a basket as parasitic, how often is it correct? (Minimizing false positives to protect legitimate customers).
* **Recall:** Out of all actual parasites, how many did the model catch?

By baselining Logistic Regression (Linear), Decision Trees (Hierarchical), Random Forests (Bagging), and Gradient Boosting (Boosting), we will let the data dictate which mathematical approach naturally fits the feature space before applying any advanced tuning.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# 1. Use the FULL dataset. No sampling is needed for these efficient algorithms.
X = ml_df[['Total_Items', 'Gross_Value', 'Avg_Item_Value']]
y = ml_df['Is_Parasite']

# Ensure the 94/6 ratio is perfectly preserved in both train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Scale the features (required for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Define the suite of algorithms to test (in their default states)
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# 4. Run the Bake-Off on all 155,000 rows
results = []
print("🧪 Running Algorithmic Baseline Tests on the Full Dataset...\n")

for name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Evaluate strictly on the minority class (Parasite = 1)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results.append({
        "Model": name,
        "Parasite Precision": round(precision, 3),
        "Parasite Recall": round(recall, 3),
        "Parasite F1-Score": round(f1, 3)
    })

# 5. Display the results ranked by Precision
results_df = pd.DataFrame(results).sort_values(by="Parasite Precision", ascending=False)
display(results_df)

🧪 Running Algorithmic Baseline Tests on the Full Dataset...



# Explanation.
Logistic Regression completely flatlined with a 0.000. It attempted to draw a straight line through a 94/6 class imbalance and failed entirely, effectively guessing that every single customer was profitable.

The Decision Tree and Random Forest tried to capture the complexity but failed the business rule. With a Precision of roughly ~26%, if we deployed them right now, 3 out of every 4 customers they penalized would be legitimate shoppers. That is the exact false-positive crisis we were trying to avoid.

Then there is Gradient Boosting.

Without any tuning, it organically achieved a 73.5% Precision. When this algorithm flags a basket as a margin-bleeding parasite, it is correct nearly 3 out of 4 times. It naturally protects our legitimate customer base.

The trade-off is its Recall (4.8%). Right now, the algorithm is so terrified of making a mistake that it only catches 1 out of every 20 parasites. It is too conservative.

We are officially dropping the Voting Ensemble. The math has proven that a single Gradient Boosting model is the superior architecture for this feature space. Our new goal is to gently tune this specific model to increase that 4.8% Recall without destroying our 73.5% Precision.

## Step 4: Gradient Boosting & Threshold Tuning

Gradient Boosting emerged as the clear mathematical winner, organically achieving a 73.5% Precision on the minority class, perfectly aligning with our business rule to protect legitimate shoppers. However, its default 4.8% Recall indicates it is overly conservative.

Instead of accepting the algorithm's default 50% confidence threshold for classification, we will extract the continuous probabilities (`predict_proba`) and manually map the optimal decision boundary to balance Precision and Recall for maximum commercial impact.

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

print("⚙️ Isolating and tuning the Gradient Boosting architecture...\n")

# 1. Isolate the winning algorithm
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)

# 2. Extract the continuous probabilities instead of the hard classifications
# We want the probability that the basket is a Parasite (Class 1)
y_probs = gb_model.predict_proba(X_test_scaled)[:, 1]

# 3. Calculate Precision and Recall across every possible confidence threshold
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# 4. Create a DataFrame to easily view the trade-offs
threshold_df = pd.DataFrame({
    'Threshold': np.append(thresholds, 1.0),
    'Precision': precisions,
    'Recall': recalls
})

# Filter for thresholds that maintain at least 50% Precision
viable_thresholds = threshold_df[threshold_df['Precision'] >= 0.50]

print("📊 Top 10 Viable Decision Thresholds (Precision >= 50%):")
display(viable_thresholds.head(10))

# 5. Plot the Precision-Recall Curve for visual analysis
plt.figure(figsize=(8, 5))
plt.plot(recalls, precisions, marker='.', color='#FF6B6B', label='Gradient Boosting')
plt.title('Precision-Recall Trade-off (Parasite Detection)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Recall (How many Parasites did we catch?)', fontsize=12, fontweight='bold')
plt.ylabel('Precision (When flagged, how often are we right?)', fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## Step 5: Deep Feature Engineering (Behavioral Indicators)

High-level basket totals failed to mathematically separate savvy shoppers from margin-bleeding cherry-pickers. To fix this, we pivot to behavioral data engineering.

By joining the line-item `transactions` with the `product` hierarchy, we extract the shopper's intent before they reach checkout. We calculate the `Department_Spread` (is it a full grocery trip or a targeted raid?), the `Private_Label_Ratio` (are they buying high-margin store brands to offset losses?), and the `Discounted_Item_Ratio` (the volume of physical items carrying a discount).

In [ ]:
print("Fetching product hierarchy and engineering behavioral features...")

# 1. Get the product data
products = get_data()['products']

# 2. Merge line-item transactions with product details
line_items = transactions.merge(products[['product_id', 'department', 'brand']], on='product_id', how='inner')

# 3. Flag line items that had any kind of discount applied
line_items['is_discounted'] = ((line_items['retail_disc'] < 0) |
                               (line_items['coupon_disc'] < 0) |
                               (line_items['coupon_match_disc'] < 0)).astype(int)

# 4. Flag line items that are Private Label (Store Brand)
line_items['is_private_label'] = (line_items['brand'].str.upper() == 'PRIVATE').astype(int)

# 5. Group back up to the Basket level to extract the behaviors
basket_behaviors = line_items.groupby('basket_id').agg(
    Department_Spread=('department', 'nunique'),
    Total_Line_Items=('product_id', 'count'),
    Discounted_Items=('is_discounted', 'sum'),
    Private_Label_Items=('is_private_label', 'sum')
).reset_index()

# 6. Calculate the Behavioral Ratios
basket_behaviors['Discounted_Item_Ratio'] = basket_behaviors['Discounted_Items'] / basket_behaviors['Total_Line_Items']
basket_behaviors['Private_Label_Ratio'] = basket_behaviors['Private_Label_Items'] / basket_behaviors['Total_Line_Items']

# 7. Merge these new behavioral features with our original target variable (Is_Parasite)
deep_features_df = ml_df[['Total_Items', 'Gross_Value', 'Avg_Item_Value', 'Is_Parasite']].merge(
    basket_behaviors[['basket_id', 'Department_Spread', 'Discounted_Item_Ratio', 'Private_Label_Ratio']],
    on=ml_df.index, # Joining on the index which represents basket_id from earlier
    how='inner'
).rename(columns={'key_0': 'basket_id'})

deep_features_df.set_index('basket_id', inplace=True)

print("✅ Behavioral Engineering Complete!")
print(f"Total Baskets Ready for Modeling: {len(deep_features_df):,}\n")

# Let's observe how these new features differ between Profitable and Parasitic baskets
print("📊 Average Behavior: Profitable (0) vs Parasitic (1)")
display(deep_features_df.groupby('Is_Parasite')[['Department_Spread', 'Discounted_Item_Ratio', 'Private_Label_Ratio']].mean().round(3))